In [2]:
here::i_am("rna/celltype_proportions/plot_celltype_proportions.R")

source(here::here("settings.R"))
source(here::here("utils.R"))

######################
## Define arguments ##
######################

args = list()
## START TEST ##
args$metadata <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$celltype_label <- "celltype.mapped_mnn"
args$outdir <- '/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/rna/celltype_proportions/celltype.mapped_mnn'
## END TEST ##

height = 8
if(args$celltype_label == 'celltype_extended.mapped_mnn'){
    opts$celltype.colors  = opts$celltype_extended.colors
    height = 14
}


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code



In [8]:

##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%opts$samples & !is.na(eval(as.name(args$celltype_label)))] %>%
  setnames(args$celltype_label,"celltype")

################################################
## Calculate cell type proportions per sample ##
################################################

to.plot <- sample_metadata %>%
  .[,N:=.N,by="sample"] %>%
  .[,.(N=.N, celltype_proportion=.N/unique(N)),by=c("sample","day","celltype")] %>%
  setorder(sample)  %>% .[,sample:=factor(sample,levels=args$samples)]


In [10]:
to.plot

sample,day,celltype,N,celltype_proportion
<fct>,<chr>,<chr>,<int>,<dbl>
NA,D3,Epiblast,410,0.1299112801
NA,D3,Anterior_Primitive_Streak,843,0.2671102662
NA,D3,Primitive_Streak,1202,0.3808618504
NA,D3,Nascent_mesoderm,153,0.0484790875
NA,D3,Def._endoderm,368,0.1166032953
NA,D3,Surface_ectoderm,79,0.0250316857
NA,D3,Paraxial_mesoderm,27,0.0085551331
NA,D3,Forebrain_Midbrain_Hindbrain,26,0.0082382763
NA,D3,Rostral_neurectoderm,25,0.0079214195


In [ ]:
#########################
## Horizontal barplots ##
#########################

# Define colours and cell type order
opts$celltype.colors <- opts$celltype.colors[names(opts$celltype.colors) %in% unique(to.plot$celltype)]
to.plot[,celltype:=factor(celltype, levels=names(opts$celltype.colors))]

for (i in unique(to.plot$day)) {
  p <- ggplot(to.plot[day==i], aes(x=celltype, y=celltype_proportion)) +
    geom_bar(aes(fill=celltype), stat="identity", color="black") +
    scale_fill_manual(values=opts$celltype.colors) +
    facet_wrap(~sample, nrow=1, scales="free_x") +
    coord_flip() +
    labs(y="Proportion of cells") +
    theme_bw() +
    theme(
      legend.position = "none",
      strip.background = element_blank(),
      strip.text = element_text(color="black", size=rel(0.5)),
      axis.title.x = element_text(color="black", size=rel(0.9)),
      axis.title.y = element_blank(),
      axis.text.y = element_text(size=rel(1), color="black"),
      axis.text.x = element_text(size=rel(1), color="black")
    )
  
  pdf(sprintf("%s/celltype_proportions_%s_horizontal_barplots_per_sample.pdf",args$outdir,i), width=10, height=height)
  print(p)
  dev.off()
}
